## Part2

In [17]:
import pandas as pd
import spacy
from spellchecker import SpellChecker
import re
import matplotlib.pyplot as plt


In [5]:
tokens = [nlp("warranty")[0], nlp("guarantee")[0]]
vocab = [w for w in nlp.vocab if w.has_vector and w.is_lower and w.is_alpha]

# Get top similar words
similar_words = set()
for token in tokens:
    similarities = sorted(vocab, key=lambda w: token.similarity(w), reverse=True)
    top_similar = [w.text for w in similarities[:20]]  # top 20 similar words
    similar_words.update(top_similar)

print(similar_words)


{'does', 'they', 'can', 'need', 'that', 'warranty', 'these', 'have', 'ca', 'may', 'we', 'would', 'all', 'or', 'not', 'w', 'ought', 'are', 'it', 'must', 'should', 'you', 'guarantee', 'wo', 'this'}


#### Select important Columns from Training dataset
#### Dropping records with out reviewText

In [10]:

df = pd.read_csv("train_data.csv")
df = df.dropna(subset=["reviewText"])
df = df[["asin", "reviewText", "overall", "vote", "verified", "reviewTime"]]


C:\Users\Asus\AppData\Local\Temp\ipykernel_31172\1522382536.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train_data.csv")


#### Transforming Text to Lemma Form
#### Removing stopwords
#### Using pipe for speed

In [14]:
nlp = spacy.load("en_core_web_sm", disable=["parser","ner"])
texts = df['clean_text'].tolist()
processed = []
for doc in nlp.pipe(texts, batch_size=1000):
    toks = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha and len(t)>2]
    processed.append(" ".join(toks))
df['processed_text'] = processed

#### Check Spellings with spellchecker()

In [18]:
spell = SpellChecker()

def correct_typos(text):
    if not isinstance(text, str):
        return ""
    corrected = []
    for word in text.split():
        fixed = spell.correction(word)
        if fixed is None:
            fixed = word  
        corrected.append(str(fixed))
    return " ".join(corrected)


In [19]:
print(df[['reviewText', 'processed_text']].sample(5))


                                               reviewText  \
500948  Definitely an excellent case with many nice fe...   
307358  I had a lot of loose DVD's. I was able to give...   
243256  Quality shooting on a budget. Only slight comp...   
598115  Recommend doing the setup via a mobile device ...   
597881  I now have these on 2 laptops and one desktop ...   

                                           processed_text  
500948  definitely excellent case nice feature get imp...  
307358  lot loose dvd able protective new home case sp...  
243256  quality shoot budget slight complaint one vide...  
598115  recommend setup mobile device easy extend wifi...  
597881  laptop desktop use wireless prior instal littl...  


In [ ]:
df.to_csv("cleaned_amazon_reviews.csv", index=False) # Our Cleaned Sataset


## NLP


#### Using en_core_web_lg model to extract similiar words to our goal word

In [1]:
nlp = spacy.load("en_core_web_lg")
tokens = [nlp("warranty")[0], nlp("guarantee")[0]]
vocab = [w for w in nlp.vocab if w.has_vector and w.is_lower and w.is_alpha]

similar_words = set()
for token in tokens:
    similarities = sorted(vocab, key=lambda w: token.similarity(w), reverse=True)
    top_similar = [w.text for w in similarities[:20]]  #
    similar_words.update(top_similar)

print(similar_words)


{'wo', 'can', 'not', 'need', 'should', 'this', 'does', 'they', 'guarantee', 'must', 'warranty', 'it', 'that', 'we', 'these', 'or', 'are', 'you', 'all', 'ca', 'may', 'w', 'have', 'ought', 'would'}


In [2]:


df = pd.read_csv("cleaned_amazon_reviews.csv")

#Using regex to select those comments with Our target words & Combine all keywords into one regex pattern
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, similar_words)) + r')\b', re.IGNORECASE)
df_filtered = df[df['reviewText'].astype(str).str.contains(pattern)]


C:\Users\Asus\AppData\Local\Temp\ipykernel_6892\1733667494.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_amazon_reviews.csv")
C:\Users\Asus\AppData\Local\Temp\ipykernel_6892\1733667494.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_filtered = df[df['reviewText'].astype(str).str.contains(pattern)]


In [3]:
avg_rating_per_product = (
    df_filtered.groupby('asin')['overall']
    .mean()
    .reset_index()
    .rename(columns={'overall': 'avg_warranty_rating'})
)

print(avg_rating_per_product.sort_values('avg_warranty_rating', ascending=False).value_counts())


asin        avg_warranty_rating
B01HJH42KU  4.000000               1
059449771X  4.333333               1
073530498X  5.000000               1
101635370   2.500000               1
1059844575  4.166667               1
                                  ..
1400699169  4.500000               1
140053271X  4.200000               1
1400501717  4.500000               1
1254875778  5.000000               1
1061861740  5.000000               1
Name: count, Length: 90551, dtype: int64


In [4]:
print(avg_rating_per_product['avg_warranty_rating'].describe())


count    90551.000000
mean         3.936369
std          1.022903
min          1.000000
25%          3.500000
50%          4.058824
75%          4.788170
max          5.000000
Name: avg_warranty_rating, dtype: float64


In [14]:
sample_products = avg_rating_per_product.sample(10)
print(sample_products)


             asin  avg_warranty_rating
83729  B01DHTWDDM             3.000000
11640  B003HC89LU             3.800000
43027  B00KCZSI5W             1.000000
79497  B01AWGXVW0             4.500000
74189  B017JIHKWQ             4.323529
1886   B0001NBHQ2             4.000000
50376  B00OFP9MNM             5.000000
35291  B00G5WY3UK             4.500000
82609  B01CX2MGOU             3.400000
76061  B018QGYEV0             4.133333


### Part3